# 马铃薯叶片疾病分类：训练 & 四模型对比

支持两种运行方式，**选一种执行对应 Section 即可**：

| | 方式 A — 本地训练 | 方式 B — Training Job |
|--|--|--|
| 环境 | 有 GPU 的 Notebook 实例（ml.g5.2xlarge）| 任意环境，GPU 在云端 |
| 执行 | 逐格运行 Section A | 逐格运行 Section B |
| 产出 | 本地 `runs/.../best.pt` | S3 `model.tar.gz`（含四个模型）|
| 下一步 | `deploy.ipynb` → `MODEL_SOURCE='local'` | `deploy.ipynb` → `MODEL_SOURCE='s3'` |

**数据结构**（两种方式相同）：
```
data/train/{Early Blight, Healthy, Late Blight}/
data/val/{Early Blight, Healthy, Late Blight}/
```
来源：[PlantVillage 马铃薯子集](https://www.kaggle.com/datasets/arjuntejaswi/plant-village)，三分类共 ~2152 张。

In [ ]:
!pip install -q ultralytics torchvision pandas matplotlib sagemaker

In [ ]:
import os, time, glob, json, warnings, pathlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

warnings.filterwarnings('ignore')

DATA_DIR    = 'data'
IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_CLASSES = 3
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# 训练哪些模型（方式 A 生效；方式 B 通过超参控制）
MODELS_TO_TRAIN = ['yolo', 'resnet50', 'mobilenet', 'cnn']  # 仅 YOLO：['yolo']

# 博文参数；快速验证可调小
YOLO_EPOCHS   = 300
RESNET_EPOCHS = 25
MOBILE_EPOCHS = 100
CNN_EPOCHS    = 100
EARLY_STOP_PAT = 8

print(f'device={DEVICE}')
try:
    classes = sorted(os.listdir(os.path.join(DATA_DIR, 'train')))
    counts  = {c: len(os.listdir(os.path.join(DATA_DIR, 'train', c))) for c in classes}
    print(f'classes={counts}')
except FileNotFoundError:
    print('⚠️  data/ 不存在，请先准备数据集')

results = {}

In [ ]:
# DataLoader + 数据增强（方式 A 的 PyTorch 模型使用）
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def make_loaders():
    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_tf)
    print(f'train={len(train_ds)}, val={len(val_ds)}, classes={train_ds.class_to_idx}')
    return (DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True))

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

class EarlyStopping:
    def __init__(self, patience=8, delta=1e-4):
        self.patience = patience; self.delta = delta
        self.best = 0.0; self.counter = 0; self.best_state = None
    def step(self, val_acc, model):
        if val_acc > self.best + self.delta:
            self.best = val_acc
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0; return False
        self.counter += 1
        return self.counter >= self.patience

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

def train_pytorch(model, train_loader, val_loader, epochs, lr, use_mixup, label):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss()
    es    = EarlyStopping(patience=EARLY_STOP_PAT)
    t0    = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = mixup_loss(crit, model(x), *mixup_data(x, y)[1:]) if use_mixup else crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
        val_acc = correct / total
        if ep % 5 == 0 or ep == 1:
            print(f'  [{label}] epoch {ep:3d}/{epochs}  val_acc={val_acc:.4f}  {time.time()-t0:.0f}s')
        if es.step(val_acc, model):
            print(f'  [{label}] 早停 @ epoch {ep}, best={es.best:.4f}'); break
    if es.best_state: model.load_state_dict(es.best_state)
    return es.best, time.time() - t0

class PotatoCNN(nn.Module):
    @staticmethod
    def _block(i, o):
        return nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.MaxPool2d(2))
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.enc  = nn.Sequential(self._block(3,32), self._block(32,64), self._block(64,128), self._block(128,256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4),
                                  nn.Linear(256,128), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(128,num_classes))
    def forward(self, x): return self.head(self.enc(x))

---
## 方式 A：本地训练（需 GPU Notebook 实例）

按顺序运行以下 Cell。训完后打开 `deploy.ipynb`，`MODEL_SOURCE='local'`。

In [ ]:
# YOLO11n-cls
if 'yolo' in MODELS_TO_TRAIN:
    from ultralytics import YOLO
    print(f'▶ YOLO11n-cls  epochs={YOLO_EPOCHS}')
    t0 = time.time()
    yolo = YOLO('yolo11n-cls.pt')
    yolo_res = yolo.train(task='classify', data=DATA_DIR,
                          epochs=YOLO_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False)
    yolo_sec = time.time() - t0
    best_pt  = pathlib.Path(yolo_res.save_dir) / 'weights' / 'best.pt'
    val_acc  = float(yolo_res.results_dict.get('metrics/accuracy_top1', 0))
    results['YOLO11n-cls'] = dict(val_acc=round(val_acc,4), train_min=round(yolo_sec/60,1),
        params_m=round(sum(p.numel() for p in yolo.model.parameters())/1e6,2),
        model_mb=round(best_pt.stat().st_size/1e6,1) if best_pt.exists() else 0,
        notes=f'epochs={YOLO_EPOCHS}')
    print(f'✓ val_acc={val_acc:.4f}  {yolo_sec/60:.1f}min  best.pt→{best_pt}')
    # 快速验证
    s = glob.glob(f'{DATA_DIR}/val/*/*')[0]
    r = YOLO(str(best_pt)).predict(s, verbose=False)[0]
    print(f'  验证: {r.names[r.probs.top1]}  {float(r.probs.top1conf):.1%}')

In [ ]:
# ResNet50
if 'resnet50' in MODELS_TO_TRAIN:
    train_loader, val_loader = make_loaders()
    print(f'▶ ResNet50  epochs={RESNET_EPOCHS}')
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
    for p in resnet.parameters(): p.requires_grad = False
    for p in resnet.fc.parameters(): p.requires_grad = True
    a1, s1 = train_pytorch(resnet, train_loader, val_loader, min(5,RESNET_EPOCHS), 1e-3, False, 'ResNet50-warmup')
    for p in resnet.parameters(): p.requires_grad = True
    a2, s2 = train_pytorch(resnet, train_loader, val_loader, RESNET_EPOCHS, 3e-4, True, 'ResNet50')
    os.makedirs('model_checkpoints', exist_ok=True)
    out = 'model_checkpoints/resnet50_best.pt'; torch.save(resnet.state_dict(), out)
    results['ResNet50'] = dict(val_acc=round(max(a1,a2),4), train_min=round((s1+s2)/60,1),
        params_m=round(count_params(resnet),2), model_mb=round(os.path.getsize(out)/1e6,1),
        notes=f'ImageNet pretrained, epochs={RESNET_EPOCHS}')
    print(f'✓ val_acc={max(a1,a2):.4f}')

In [ ]:
# MobileNetV3
if 'mobilenet' in MODELS_TO_TRAIN:
    if 'train_loader' not in dir(): train_loader, val_loader = make_loaders()
    print(f'▶ MobileNetV3  epochs={MOBILE_EPOCHS}')
    mob = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    mob.classifier[-1] = nn.Linear(mob.classifier[-1].in_features, NUM_CLASSES)
    acc, sec = train_pytorch(mob, train_loader, val_loader, MOBILE_EPOCHS, 3e-4, True, 'MobileNetV3')
    os.makedirs('model_checkpoints', exist_ok=True)
    out = 'model_checkpoints/mobilenet_best.pt'; torch.save(mob.state_dict(), out)
    results['MobileNetV3'] = dict(val_acc=round(acc,4), train_min=round(sec/60,1),
        params_m=round(count_params(mob),2), model_mb=round(os.path.getsize(out)/1e6,1),
        notes=f'ImageNet pretrained, epochs={MOBILE_EPOCHS}')
    print(f'✓ val_acc={acc:.4f}')

In [ ]:
# 自定义 CNN
if 'cnn' in MODELS_TO_TRAIN:
    if 'train_loader' not in dir(): train_loader, val_loader = make_loaders()
    print(f'▶ PotatoCNN  epochs={CNN_EPOCHS}')
    cnn = PotatoCNN()
    acc, sec = train_pytorch(cnn, train_loader, val_loader, CNN_EPOCHS, 1e-3, True, 'PotatoCNN')
    os.makedirs('model_checkpoints', exist_ok=True)
    out = 'model_checkpoints/cnn_best.pt'; torch.save(cnn.state_dict(), out)
    results['PotatoCNN'] = dict(val_acc=round(acc,4), train_min=round(sec/60,1),
        params_m=round(count_params(cnn),2), model_mb=round(os.path.getsize(out)/1e6,1),
        notes=f'从零训练, epochs={CNN_EPOCHS}')
    print(f'✓ val_acc={acc:.4f}')

In [ ]:
# 对比结果
if results:
    df = (pd.DataFrame(results).T.reset_index().rename(columns={'index':'model'})
          .sort_values('val_acc', ascending=False).reset_index(drop=True))
    df['val_acc_pct'] = df['val_acc'].apply(lambda x: f'{float(x)*100:.2f}%')
    print('\n=== 四模型对比 ===')
    print(df[['model','val_acc_pct','train_min','params_m','model_mb']].to_string(index=False))
    os.makedirs('model_checkpoints', exist_ok=True)
    df.to_csv('model_checkpoints/comparison.csv', index=False)
    with open('model_checkpoints/comparison.json','w',encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(11,4))
        names  = df['model'].tolist()
        accs   = [float(v)*100 for v in df['val_acc'].tolist()]
        colors = ['#2ecc71' if m=='YOLO11n-cls' else '#3498db' for m in names]
        axes[0].bar(names, accs, color=colors)
        axes[0].set_title('验证集准确率 (%)')
        axes[0].set_ylim(max(0,min(accs)*0.95), 101)
        for i,v in enumerate(accs): axes[0].text(i,v+.2,f'{v:.1f}%',ha='center',fontsize=9)
        times = [float(v) for v in df['train_min'].tolist()]
        axes[1].bar(names, times, color=colors)
        axes[1].set_title('训练时间 (min)')
        for i,v in enumerate(times): axes[1].text(i,v+.3,f'{v:.0f}',ha='center',fontsize=9)
        plt.tight_layout()
        plt.savefig('model_checkpoints/comparison.png', dpi=120, bbox_inches='tight')
        plt.show()
    except ImportError: pass
    print('\n下一步：deploy.ipynb → MODEL_SOURCE="local"')

---
## 方式 B：SageMaker Training Job（无本地 GPU）

在 **ml.g5.2xlarge** 上跑 `sagemaker/train_job.py`，训完把模型写回 S3。  
训完后打开 `deploy.ipynb`，将 Cell 1 的 `MODEL_SOURCE='s3'` 并填入下面打印的 `MODEL_DATA_S3`。

In [ ]:
import sagemaker, boto3
from sagemaker.pytorch import PyTorch
from sagemaker.inputs import TrainingInput

sess   = sagemaker.Session()
role   = sagemaker.get_execution_role()
bucket = sess.default_bucket()

DATA_S3   = f's3://{bucket}/potato-demo/data'
OUTPUT_S3 = f's3://{bucket}/potato-demo/training-output'
print(f'data   → {DATA_S3}')
print(f'output → {OUTPUT_S3}')

In [ ]:
# 上传本地 data/ 到 S3（已上传可跳过）
import os
LOCAL_DATA = 'data'
if os.path.isdir(LOCAL_DATA):
    s3 = boto3.client('s3'); n = 0
    for root, _, files in os.walk(LOCAL_DATA):
        for fname in files:
            if fname.lower().endswith(('.jpg','.jpeg','.png')):
                lp = os.path.join(root, fname)
                s3.upload_file(lp, bucket, f'potato-demo/{os.path.relpath(lp, os.path.dirname(LOCAL_DATA))}')
                n += 1
                if n % 200 == 0: print(f'  已上传 {n} 张...')
    print(f'上传完成 {n} 张')
else:
    print(f'本地 {LOCAL_DATA}/ 不存在，跳过（假设数据已在 S3）')

In [ ]:
# 提交 Training Job
estimator = PyTorch(
    entry_point='train_job.py',
    source_dir='../sagemaker',
    role=role,
    framework_version='2.1', py_version='py310',
    instance_type='ml.g5.2xlarge',
    instance_count=1,
    hyperparameters={
        'models':              'yolo,resnet50,mobilenet,cnn',
        'yolo-epochs':         300,
        'resnet-epochs':       25,
        'mobile-epochs':       100,
        'cnn-epochs':          100,
        'img-size':            224,
        'batch-size':          16,
        'early-stop-patience': 8,
    },
    output_path=OUTPUT_S3,
    base_job_name='potato-disease-train',
    keep_alive_period_in_seconds=1800,
)
estimator.fit(
    inputs={'data': TrainingInput(DATA_S3, content_type='application/x-image')},
    wait=True, logs='All',
)
MODEL_DATA_S3 = estimator.model_data
print(f'\n✓ 训练完成')
print(f'MODEL_DATA_S3 = "{MODEL_DATA_S3}"')
print('→ 复制上面路径，填入 deploy.ipynb 的 MODEL_DATA_S3 变量')

In [ ]:
# 下载并展示四模型对比结果
import json, tarfile, tempfile
from urllib.parse import urlparse

parsed = urlparse(MODEL_DATA_S3)
s3 = boto3.client('s3')
with tempfile.TemporaryDirectory() as tmp:
    local_tar = f'{tmp}/model.tar.gz'
    s3.download_file(parsed.netloc, parsed.path.lstrip('/'), local_tar)
    with tarfile.open(local_tar) as tf: tf.extractall(tmp)
    cmp = json.load(open(f'{tmp}/comparison.json'))
df = (pd.DataFrame(cmp).T.reset_index().rename(columns={'index':'model'})
      .sort_values('val_acc', ascending=False))
df['val_acc_pct'] = df['val_acc'].apply(lambda x: f'{float(x)*100:.2f}%')
print('=== 四模型对比结果 ===')
print(df[['model','val_acc_pct','train_min','params_m','model_mb']].to_string(index=False))